# V024: temporal-convolution/LSTM hybrid

A small residual temporal adapter precedes the successful deep curve-loss LSTM. It learns how to combine neighboring known inputs before the recurrent layers. Spans3 and7 are separate arms. The adapter is zero initialized, so the initial function and backbone RNG match the original LSTM exactly. The unchanged backbone, pressure objective, features, scalers and rates isolate the added temporal representation.

The adapter uses zero-padded shifted linear maps and SiLU, avoiding the previously measured slow full Conv1d training path. Known R/C, phase, time and step outputs, plus the internal joint code, are preserved at every step. Context never crosses breaths; future known inputs are permitted by the offline task. Pressure labels are never inputs. This component does not fit or read any dataset by itself.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class V024TemporalResidualRNN(nn.Module):
    """Zero-initialized nonlinear temporal adapter before an unchanged V5 backbone."""
    def __init__(self,spec,n_features,feature_names,span):
        super().__init__();self.n_features=int(n_features);self.span=int(span)
        self.feature_names=list(feature_names)
        if self.span not in (3,7) or len(self.feature_names)!=self.n_features or len(set(self.feature_names))!=self.n_features:
            raise ValueError('V024 requires span3/7 and a unique complete feature schema')
        if not {'R','C','u_out','time_step','step'}.issubset(self.feature_names):
            raise ValueError('Known lung/phase/time channels must be named')
        # Construct the backbone first; subsequent zero parameters consume no RNG.
        self.backbone=V5PressureRNN(spec,self.n_features)
        self.context_weight=nn.Parameter(torch.zeros(self.span,self.n_features,self.n_features))
        mask=torch.tensor([0. if n in {'R','C','u_out','time_step','step'} else 1. for n in self.feature_names])
        self.register_buffer('context_output_mask',mask.reshape(1,1,-1))
        self.offsets=tuple(range(-self.span//2+1,self.span//2+1))
        if len(self.offsets)!=self.span:raise RuntimeError('Temporal offsets changed')
    @staticmethod
    def shifted(x,offset):
        if offset==0:return x
        if offset<0:
            d=-offset;return F.pad(x[:,:-d,:],(0,0,d,0))
        return F.pad(x[:,offset:,:],(0,0,0,offset))
    def context_values(self,numeric):
        result=torch.zeros_like(numeric)
        for index,offset in enumerate(self.offsets):
            result=result+F.linear(self.shifted(numeric,offset),self.context_weight[index])
        return result
    def adapt_inputs(self,x):
        if x.ndim!=3 or x.shape[1]!=80 or x.shape[-1]!=self.n_features+1:
            raise ValueError('Expected independent80-step standardized inputs plus internal lung code')
        numeric=x[...,:self.n_features]
        adapted=numeric+F.silu(self.context_values(numeric))*self.context_output_mask
        return torch.cat([adapted,x[...,-1:]],dim=-1)
    def forward(self,x):return self.backbone(self.adapt_inputs(x))

def v24_conv3_factory(spec,n_features,prep):
    return V024TemporalResidualRNN(spec,n_features,prep['feature_names'],3)
def v24_conv7_factory(spec,n_features,prep):
    return V024TemporalResidualRNN(spec,n_features,prep['feature_names'],7)
def v24_plain_factory(spec,n_features,prep):return V5PressureRNN(spec,n_features)
def v24_frozen_conv3_factory(spec,n_features,prep):
    model=v24_conv3_factory(spec,n_features,prep);model.context_weight.requires_grad_(False);return model

def convolution_selfcheck_v024(device='cpu'):
    checks={}
    def check(k,v):
        if not bool(v):raise AssertionError(k)
        checks[k]=True
    selected=torch.device(device)
    names=['R','C','u_in','u_out','time_step','step','delta_u_in']
    spec=dict(family='lstm',hidden=6,layers=2,dropout=.2,bidirectional=True,
        joint_embedding_dim=0,rc_heads=False,residual=True,head_width=4,widths=[6,4],output_concat=True)
    seed_sequence(20262401)
    x=torch.randn(3,80,len(names)+1,device=selected);x[...,-1]=torch.arange(3,device=selected)[:,None]
    for span,factory in [(3,v24_conv3_factory),(7,v24_conv7_factory)]:
        seed_sequence(303);base=V5PressureRNN(spec,len(names));base_rng=torch.get_rng_state().clone()
        seed_sequence(303);model=factory(spec,len(names),{'feature_names':names});model_rng=torch.get_rng_state().clone()
        check(str(span)+':construction_consumes_no_extra_RNG',torch.equal(base_rng,model_rng))
        check(str(span)+':identical_initial_backbone',all(torch.equal(v,model.backbone.state_dict()[k]) for k,v in base.state_dict().items()))
        base=base.to(selected);model=model.to(selected);base.eval();model.eval()
        check(str(span)+':exact_identity_inputs',torch.equal(model.adapt_inputs(x),x))
        check(str(span)+':exact_initial_prediction',torch.equal(model(x),base(x)))
        base.train();model.train()
        seed_sequence(749);a=base(x);seed_sequence(749);b=model(x)
        check(str(span)+':same_initial_dropout_function',torch.equal(a,b))
        model.zero_grad(set_to_none=True);b.square().mean().backward()
        grad=model.context_weight.grad
        check(str(span)+':adapter_has_gradient',grad is not None and bool((grad!=0).any()) and torch.isfinite(grad).all())
        protected=[names.index(n) for n in ['R','C','u_out','time_step','step']]
        check(str(span)+':protected_output_rows_have_zero_gradient',bool((grad[:,protected,:]==0).all()))
        # Independent dense Conv1d cross-correlation, used only as a tiny proof reference.
        q=torch.randn(2,80,len(names),dtype=torch.float64,device=selected,requires_grad=True)
        w=torch.randn(span,len(names),len(names),dtype=torch.float64,device=selected,requires_grad=True)
        manual=torch.zeros_like(q)
        for j,offset in enumerate(model.offsets):manual=manual+F.linear(model.shifted(q,offset),w[j])
        native=F.conv1d(q.transpose(1,2),w.permute(1,2,0),padding=span//2).transpose(1,2)
        probe=torch.randn_like(manual)
        gm=torch.autograd.grad((manual*probe).sum(),(q,w),retain_graph=True)
        gn=torch.autograd.grad((native*probe).sum(),(q,w))
        check(str(span)+':convolution_output_equivalent',torch.allclose(manual,native,rtol=1e-12,atol=1e-12))
        check(str(span)+':convolution_input_and_weight_gradients',all(torch.allclose(a,b,rtol=1e-12,atol=1e-12) for a,b in zip(gm,gn)))
        with torch.no_grad():model.context_weight.copy_(torch.randn_like(model.context_weight)*.01)
        adapted=model.adapt_inputs(x)
        check(str(span)+':protected_known_inputs_and_internal_code',torch.equal(adapted[...,-1],x[...,-1]) and torch.equal(adapted[...,protected],x[...,protected]))
        # A change in another breath cannot affect this breath's adapted sequence.
        changed=x.clone();changed[1:,:,:len(names)]+=100
        check(str(span)+':no_cross_breath_context',torch.equal(model.adapt_inputs(changed)[0],adapted[0]))
        model.eval()
        check(str(span)+':memory_resets_between_calls',torch.equal(model(x),model(x)))
        z=torch.arange(80,dtype=x.dtype,device=selected).reshape(1,80,1)
        check(str(span)+':padding_does_not_wrap',model.shifted(z,1)[0,-1,0]==0 and model.shifted(z,-1)[0,0,0]==0)
    try:V024TemporalResidualRNN(spec,len(names),names,4)
    except ValueError:checks['even_span_rejected']=True
    else:raise AssertionError('even_span_rejected')
    return checks
